<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Cuadernos/02_Desambiguacion_Etiquetado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Desambiguación y etiquetado de palabras</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Introducción</span>](#introduccion)
- [<span style="color: #005C99;">Etiquetado gramatical (POS tagging)</span>](#etiquetado-gramatical-pos-tagging)
- [<span style="color: #005C99;">El etiquetado como problema de aprendizaje</span>](#el-etiquetado-como-problema-de-aprendizaje)
- [<span style="color: #005C99;">Reconocimiento de entidades nombradas</span>](#reconocimiento-de-entidades-nombradas)
- [<span style="color: #005C99;">Desambiguación del sentido de las palabras</span>](#desambiguacion-del-sentido-de-las-palabras)
- [<span style="color: #005C99;">Algoritmo de Lesk</span>](#algoritmo-de-lesk)
- [<span style="color: #005C99;">Referencias</span>](#referencias)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [1]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib nltk spacy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Modelo de spaCy para español
import sys
!{sys.executable} -m spacy download es_core_news_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')


In [3]:
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
import spacy
from spacy import displacy

SEED = 42
np.random.seed(SEED)

for recurso in ["punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger_eng",
                "universal_tagset", "treebank", "wordnet", "omw-2.0"]:
    nltk.download(recurso, quiet=True)

nlp = spacy.load("es_core_news_sm")

<a id="introduccion"></a>
<h2 style="color: #007ACC;">Introducción</h2>

En el cuaderno anterior convertimos el texto en una secuencia de tokens. Pero no todos los tokens cumplen el mismo papel: algunos nombran cosas, otros describen acciones, otros conectan ideas. Y una misma palabra puede tener **distintas funciones y significados** según el contexto:

- *"**Bajo** el **bajo** **bajo** el puente"*: preposición, sustantivo (instrumento) y adjetivo.
- *"Me senté en el **banco** del parque"* vs. *"Abrí una cuenta en el **banco**"*.

En este cuaderno veremos tres tareas que enriquecen cada token con información lingüística:

| Tarea | Pregunta | Salida |
|-------|----------|--------|
| **Etiquetado gramatical** (POS) | ¿Qué categoría gramatical tiene cada palabra? | `NOUN`, `VERB`, `ADJ`... |
| **Reconocimiento de entidades** (NER) | ¿Qué fragmentos nombran personas, lugares u organizaciones? | `PER`, `LOC`, `ORG`... |
| **Desambiguación del sentido** (WSD) | ¿Cuál de los significados de la palabra se usa aquí? | Un sentido de un diccionario (WordNet) |

Las tres son útiles por sí mismas (extracción de información, anonimización, búsqueda) y como **características** para otras tareas.

<a id="etiquetado-gramatical-pos-tagging"></a>
<h2 style="color: #007ACC;">Etiquetado gramatical (POS tagging)</h2>

El **etiquetado gramatical** (*part-of-speech tagging*) asigna a cada token su categoría gramatical. El estándar multilingüe más usado es el conjunto de etiquetas de **Universal Dependencies**:

| Etiqueta | Categoría | Ejemplos |
|----------|-----------|----------|
| `NOUN` | Sustantivo | *casa, perro, idea* |
| `PROPN` | Nombre propio | *Colombia, María* |
| `VERB` | Verbo | *correr, comí* |
| `AUX` | Verbo auxiliar | *ha (comido), es, estaba* |
| `ADJ` | Adjetivo | *grande, rojo* |
| `ADV` | Adverbio | *rápidamente, muy* |
| `PRON` | Pronombre | *yo, ella, que* |
| `DET` | Determinante | *el, una, este* |
| `ADP` | Preposición | *en, de, bajo* |
| `CCONJ` / `SCONJ` | Conjunción coordinante / subordinante | *y, pero* / *que, porque* |
| `NUM` | Numeral | *tres, 2024* |
| `PUNCT` | Puntuación | *. , ¡* |

spaCy etiqueta en español e incluye además rasgos morfológicos (género, número, tiempo verbal...):

In [4]:
doc = nlp("Bajo el bajo bajo el puente mientras la vieja cierra la puerta.")

pd.DataFrame(
    [(t.text, t.pos_, spacy.explain(t.pos_), str(t.morph)) for t in doc],
    columns=["token", "POS", "descripción", "rasgos morfológicos"],
)

,token,POS,descripción,rasgos morfológicos
0,Bajo,ADP,adposition,
1,el,DET,determiner,Definite=Def|Gender=Masc|Number=Sing|PronType=Art
2,bajo,NOUN,noun,Gender=Masc|Number=Sing
3,bajo,ADP,adposition,
4,el,DET,determiner,Definite=Def|Gender=Masc|Number=Sing|PronType=Art
5,puente,NOUN,noun,Gender=Masc|Number=Sing
6,mientras,SCONJ,subordinating conjunction,
7,la,DET,determiner,Definite=Def|Gender=Fem|Number=Sing|PronType=Art
8,vieja,ADJ,adjective,Gender=Fem|Number=Sing
9,cierra,VERB,verb,Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbF...


La palabra *bajo* aparece tres veces con tres funciones distintas: verbo (*yo bajo*), sustantivo (el instrumento) y preposición. Además, *vieja* es aquí un sustantivo (la señora) y no un adjetivo. El etiquetador debe usar el **contexto** (las palabras vecinas) para decidir, y no siempre acierta: revisa la tabla y observa que el modelo pequeño confunde el primer *Bajo* y *vieja*. Las oraciones así de rebuscadas son difíciles incluso para modelos entrenados.

<h3 style="color: #007ACC;">¿Cómo etiqueta spaCy?</h3>

`spacy.explain()` no etiqueta nada: solo busca la etiqueta en un glosario y devuelve su descripción (`"ADP"` → `"adposition"`). Las etiquetas ya estaban en `t.pos_` desde que ejecutamos `nlp(...)`, que pasa el texto por un **pipeline** de componentes:

In [5]:
print(nlp.pipe_names)

['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


En los modelos de español, la categoría gramatical y los rasgos morfológicos los predice el componente `morphologizer`, que es una **red neuronal**:

1. `tok2vec` convierte cada token en un vector: combina *embeddings* de la palabra, su prefijo, su sufijo y su forma ortográfica (mayúsculas, dígitos...), y los pasa por varias capas **convolucionales** que mezclan la información de las palabras vecinas.
2. `morphologizer` aplica una capa **softmax** sobre ese vector para elegir la combinación de categoría y rasgos más probable (por ejemplo `NOUN` + `Gender=Masc|Number=Sing`).

Es decir, cada token se clasifica mirando una **ventana de contexto**, aprendida a partir de un corpus anotado del español.

En inglés, NLTK trae un etiquetador distinto, basado en un **perceptrón promediado** (un modelo lineal con características diseñadas a mano: la palabra, sus sufijos, las etiquetas anteriores...). Es una adaptación del etiquetador que usaba spaCy en su primera versión, antes de pasar a redes neuronales. Por defecto usa las etiquetas del Penn Treebank, pero podemos pedirle el conjunto universal:

In [6]:
from nltk import pos_tag, word_tokenize

for oracion in ["I saw her duck under the table", "Time flies like an arrow", "The old man the boats"]:
    print(pos_tag(word_tokenize(oracion), tagset="universal"))

[('I', 'PRON'), ('saw', 'VERB'), ('her', 'PRON'), ('duck', 'NOUN'), ('under', 'ADP'), ('the', 'DET'), ('table', 'NOUN')]
[('Time', 'NOUN'), ('flies', 'NOUN'), ('like', 'ADP'), ('an', 'DET'), ('arrow', 'NOUN')]
[('The', 'DET'), ('old', 'ADJ'), ('man', 'NOUN'), ('the', 'DET'), ('boats', 'NOUN')]


- *duck* puede ser verbo (*agacharse*) o sustantivo (*pato*).
- *Time flies like an arrow* tiene varias lecturas gramaticalmente válidas.
- *The old man the boats* es una oración "de vía muerta" (*garden path*): *man* es el verbo (*los viejos tripulan los botes*). Casi cualquier etiquetador la etiqueta mal.

<a id="el-etiquetado-como-problema-de-aprendizaje"></a>
<h2 style="color: #007ACC;">El etiquetado como problema de aprendizaje</h2>

¿Cómo aprende un modelo a etiquetar? Necesitamos un **corpus anotado**: oraciones donde cada palabra ya tiene su etiqueta correcta. NLTK incluye una muestra del **Penn Treebank** (≈100 000 palabras de noticias del *Wall Street Journal*).

Construyamos la línea base más simple posible: **asignar a cada palabra la etiqueta con la que aparece más veces en entrenamiento**. Para palabras desconocidas usamos la etiqueta más común en general.

In [7]:
from nltk.corpus import treebank

# El Treebank incluye "huellas" sintácticas sin texto real (*T*-1, 0...) con la etiqueta X: las quitamos
oraciones = [[(p, e) for p, e in o if e != "X"] for o in treebank.tagged_sents(tagset="universal")]
corte = int(0.8 * len(oraciones))
train_sents, test_sents = oraciones[:corte], oraciones[corte:]

print(f"Oraciones: {len(train_sents)} entrenamiento, {len(test_sents)} prueba")
print(train_sents[0])

Oraciones: 3131 entrenamiento, 783 prueba
[('Pierre', 'NOUN'), ('Vinken', 'NOUN'), (',', '.'), ('61', 'NUM'), ('years', 'NOUN'), ('old', 'ADJ'), (',', '.'), ('will', 'VERB'), ('join', 'VERB'), ('the', 'DET'), ('board', 'NOUN'), ('as', 'ADP'), ('a', 'DET'), ('nonexecutive', 'ADJ'), ('director', 'NOUN'), ('Nov.', 'NOUN'), ('29', 'NUM'), ('.', '.')]


In [8]:
conteo_palabra_etiqueta = defaultdict(Counter)
for oracion in train_sents:
    for palabra, etiqueta in oracion:
        conteo_palabra_etiqueta[palabra.lower()][etiqueta] += 1

etiqueta_mas_comun = Counter(e for o in train_sents for _, e in o).most_common(1)[0][0]
etiquetador_base = {p: c.most_common(1)[0][0] for p, c in conteo_palabra_etiqueta.items()}


def etiquetar_base(palabras):
    return [etiquetador_base.get(p.lower(), etiqueta_mas_comun) for p in palabras]


def exactitud(etiquetador, oraciones):
    aciertos = total = 0
    for oracion in oraciones:
        palabras, reales = zip(*oracion)
        predichas = etiquetador(list(palabras))
        aciertos += sum(p == r for p, r in zip(predichas, reales))
        total += len(reales)
    return aciertos / total


print(f"Etiqueta por defecto para palabras desconocidas: {etiqueta_mas_comun}")
print(f"Exactitud de la línea base: {exactitud(etiquetar_base, test_sents):.2%}")

Etiqueta por defecto para palabras desconocidas: NOUN
Exactitud de la línea base: 92.46%


¡Más del 92 % con una regla trivial! Esto revela algo importante: **la mayoría de las palabras no son ambiguas**, o casi siempre se usan con la misma etiqueta. El reto está en el resto: palabras ambiguas y palabras nunca vistas. Veamos dónde falla:

In [9]:
vocab_train = set(conteo_palabra_etiqueta)
errores = Counter()
conocidas = desconocidas = aciertos_conocidas = aciertos_desconocidas = 0

for oracion in test_sents:
    palabras, reales = zip(*oracion)
    for p, pred, real in zip(palabras, etiquetar_base(list(palabras)), reales):
        if p.lower() in vocab_train:
            conocidas += 1
            aciertos_conocidas += pred == real
        else:
            desconocidas += 1
            aciertos_desconocidas += pred == real
        if pred != real:
            errores[(real, pred)] += 1

print(f"Palabras conocidas:   {aciertos_conocidas / conocidas:.2%} de exactitud ({conocidas} tokens)")
print(f"Palabras desconocidas: {aciertos_desconocidas / desconocidas:.2%} de exactitud ({desconocidas} tokens)")
pd.DataFrame([(r, p, n) for (r, p), n in errores.most_common(8)], columns=["real", "predicha", "errores"])

Palabras conocidas:   96.09% de exactitud (16993 tokens)
Palabras desconocidas: 56.39% de exactitud (1706 tokens)


,real,predicha,errores
0,VERB,NOUN,371
1,NUM,NOUN,265
2,ADJ,NOUN,264
3,NOUN,ADJ,147
4,NOUN,VERB,76
5,DET,ADP,57
6,ADV,ADJ,34
7,ADV,PRT,33


El problema principal son las **palabras desconocidas**: casi la mitad se etiquetan mal porque todas reciben `NOUN`. Muchos errores `NUM → NOUN` son números que no aparecieron en entrenamiento; reglas simples sobre la forma de la palabra (¿es un número?, ¿termina en *-ed* o *-ly*?, ¿empieza con mayúscula?) los corregirían.

Comparemos con el etiquetador preentrenado de NLTK, que usa el contexto (palabras y etiquetas vecinas, sufijos, mayúsculas):

In [10]:
def etiquetar_nltk(palabras):
    return [e for _, e in pos_tag(palabras, tagset="universal")]


print(f"Exactitud del perceptrón de NLTK: {exactitud(etiquetar_nltk, test_sents):.2%}")

Exactitud del perceptrón de NLTK: 97.28%


> ⚠️ Esta comparación no es del todo justa: el etiquetador de NLTK fue entrenado con el Penn Treebank completo, que probablemente incluye nuestras oraciones de prueba. Aun así ilustra el techo al que llegan los buenos etiquetadores (≈97 %).

El salto de la línea base al ≈97 % proviene de **usar el contexto**. En el módulo 2 formalizaremos esta idea con los **modelos ocultos de Markov (HMM)** y el **algoritmo de Viterbi**, que eligen la secuencia de etiquetas más probable para toda la oración.

<a id="reconocimiento-de-entidades-nombradas"></a>
<h2 style="color: #007ACC;">Reconocimiento de entidades nombradas</h2>

El **reconocimiento de entidades nombradas** (*NER*) identifica fragmentos de texto que nombran entidades del mundo real y los clasifica. Los modelos de spaCy en español reconocen:

| Etiqueta | Entidad |
|----------|---------|
| `PER` | Persona |
| `LOC` | Lugar |
| `ORG` | Organización |
| `MISC` | Otras (eventos, obras, nacionalidades...) |

In [11]:
texto = (
    "Gabriel García Márquez nació en Aracataca y estudió Derecho en la Universidad Nacional de Colombia. "
    "En 1982 recibió el Premio Nobel de Literatura en Estocolmo por Cien años de soledad."
)
doc = nlp(texto)

print(pd.DataFrame([(e.text, e.label_, e.start_char, e.end_char) for e in doc.ents],
                   columns=["entidad", "tipo", "inicio", "fin"]))
displacy.render(doc, style="ent", jupyter=True)

                            entidad  tipo  inicio  fin
0            Gabriel García Márquez   PER       0   22
1                         Aracataca   LOC      32   41
2                           Derecho   PER      52   59
3  Universidad Nacional de Colombia   LOC      66   98
4        Premio Nobel de Literatura  MISC     119  145
5                         Estocolmo   LOC     149  158
6                              Cien   LOC     163  167

El modelo pequeño acierta con las personas y los lugares, pero comete errores instructivos: etiqueta *Derecho* (la carrera) como persona por la mayúscula inicial, clasifica la universidad como lugar en vez de organización y toma *Cien* como un lugar, cuando *Cien años de soledad* es una obra (`MISC`). NER depende mucho del dominio y del tamaño del modelo.

Aunque las entidades abarcan varias palabras, internamente NER se resuelve como **etiquetado de secuencias** token a token, igual que POS. Para ello se usa el esquema **BIO**:

- `B-TIPO`: comienzo (*begin*) de una entidad.
- `I-TIPO`: continuación (*inside*) de una entidad.
- `O`: fuera (*outside*) de cualquier entidad.

In [12]:
pd.DataFrame(
    [(t.text, f"{t.ent_iob_}-{t.ent_type_}" if t.ent_type_ else "O") for t in doc[:16]],
    columns=["token", "etiqueta BIO"],
).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
token,Gabriel,García,Márquez,nació,en,Aracataca,y,estudió,Derecho,en,la,Universidad,Nacional,de,Colombia,.
etiqueta BIO,B-PER,I-PER,I-PER,O,O,B-LOC,O,O,B-PER,O,O,B-LOC,I-LOC,I-LOC,I-LOC,O


Aplicaciones típicas de NER: **anonimizar** documentos (reemplazar nombres de personas en historias clínicas), extraer información de noticias o contratos, enlazar menciones con una base de conocimiento.

In [13]:
def anonimizar(texto, tipos=("PER",)):
    doc = nlp(texto)
    resultado = texto
    # Reemplazamos de atrás hacia adelante para no desplazar los índices
    for ent in reversed(doc.ents):
        if ent.label_ in tipos:
            resultado = resultado[:ent.start_char] + f"[{ent.label_}]" + resultado[ent.end_char:]
    return resultado


print(anonimizar("La paciente Ana María Rodríguez fue atendida por el doctor Carlos Gómez en Tunja."))

La paciente [PER] fue atendida por el doctor [PER] en Tunja.


<a id="desambiguacion-del-sentido-de-las-palabras"></a>
<h2 style="color: #007ACC;">Desambiguación del sentido de las palabras</h2>

La **desambiguación del sentido de las palabras** (*Word Sense Disambiguation*, WSD) decide cuál de los significados de una palabra polisémica se usa en un contexto dado.

Para eso necesitamos un inventario de sentidos. El más usado es **WordNet**, una base de datos léxica del inglés que agrupa palabras en **synsets** (conjuntos de sinónimos que representan un concepto), cada uno con una definición, ejemplos y relaciones con otros synsets (hiperónimos, hipónimos, antónimos...). El *Open Multilingual WordNet* enlaza esos synsets con palabras de otros idiomas, incluido el español.

In [14]:
from nltk.corpus import wordnet as wn

sentidos = wn.synsets("bank")
print(f"'bank' tiene {len(sentidos)} sentidos en WordNet. Los primeros:\n")
for s in sentidos[:6]:
    print(f"{s.name():<22} {s.definition()}")

'bank' tiene 18 sentidos en WordNet. Los primeros:

bank.n.01              sloping land (especially the slope beside a body of water)
depository_financial_institution.n.01 a financial institution that accepts deposits and channels the money into lending activities
bank.n.03              a long ridge or pile
bank.n.04              an arrangement of similar objects in a row or in tiers
bank.n.05              a supply or stock held in reserve for future use (especially in emergencies)
bank.n.06              the funds held by a gambling house or the dealer in some gambling games


In [15]:
# Sentidos de "banco" en español a través del Open Multilingual WordNet
for s in wn.synsets("banco", lang="spa"):
    print(f"{s.name():<22} {s.lemma_names('spa')[:4]}  ->  {s.definition()}")

bank.n.09              ['banco']  ->  a building in which the business of banking transacted
bench.n.01             ['banco', 'banquillo']  ->  a long seat for more than one person
pew.n.01               ['banco', 'banco_de_iglesia']  ->  long bench with backs; used in church by the congregation
school.n.07            ['banco', 'cardumen']  ->  a large group of fish
depository_financial_institution.n.01 ['banca', 'banco']  ->  a financial institution that accepts deposits and channels the money into lending activities
bank.n.03              ['bajío', 'banco']  ->  a long ridge or pile
bar.n.08               ['bajío', 'bajo', 'banco', 'barra']  ->  a submerged (or partly submerged) ridge in a river or along a shore
sandbank.n.01          ['banco', 'banco_de_arena', 'barbada', 'médano']  ->  a submerged bank of sand near a shore or in a river; can be exposed at low tide
terrace.n.02           ['bancal', 'banco', 'terraza']  ->  a level shelf of land interrupting a declivity (with steep s

WordNet organiza los conceptos en una jerarquía. Recorrer los **hiperónimos** (conceptos más generales) muestra cómo cada sentido pertenece a una rama distinta:

In [16]:
for nombre in ["bank.n.01", "bank.n.02"]:
    camino = wn.synset(nombre).hypernym_paths()[0]
    print(" -> ".join(s.name().split(".")[0] for s in camino))

entity -> physical_entity -> object -> geological_formation -> slope -> bank
entity -> abstraction -> group -> social_group -> organization -> institution -> financial_institution -> depository_financial_institution


<a id="algoritmo-de-lesk"></a>
<h2 style="color: #007ACC;">Algoritmo de Lesk</h2>

El **algoritmo de Lesk** (1986) es el método clásico de WSD basado en diccionario. Su idea: *el sentido correcto es el que tiene la definición que más palabras comparte con el contexto.*

Versión simplificada:

1. Para cada sentido $s$ de la palabra ambigua, construir su **firma**: las palabras de su definición y ejemplos.
2. Calcular el **solapamiento** entre la firma y las palabras del contexto (sin stopwords).
3. Elegir el sentido con mayor solapamiento.

$$\hat{s} = \arg\max_{s \in \text{sentidos}(w)} \left| \text{firma}(s) \cap \text{contexto}(w) \right|$$

In [17]:
from nltk.corpus import stopwords

STOP_EN = set(stopwords.words("english"))


def palabras_contenido(texto):
    return {w.lower() for w in word_tokenize(texto) if w.isalpha() and w.lower() not in STOP_EN}


def lesk_simplificado(contexto, palabra, pos=None):
    contexto = palabras_contenido(contexto) - {palabra}
    mejor, mejor_solape = None, -1
    for sentido in wn.synsets(palabra, pos=pos):
        firma = palabras_contenido(sentido.definition() + " " + " ".join(sentido.examples()))
        solape = len(firma & contexto)
        if solape > mejor_solape:
            mejor, mejor_solape = sentido, solape
    return mejor, mejor_solape


for oracion in ["I went to the bank to deposit money and pay my loan",
                "We sat on the bank of the river and watched the water flow"]:
    sentido, solape = lesk_simplificado(oracion, "bank", pos=wn.NOUN)
    print(f"{oracion}\n  -> {sentido.name()} (solapamiento={solape}): {sentido.definition()}\n")

I went to the bank to deposit money and pay my loan
  -> depository_financial_institution.n.01 (solapamiento=1): a financial institution that accepts deposits and channels the money into lending activities

We sat on the bank of the river and watched the water flow
  -> bank.n.01 (solapamiento=4): sloping land (especially the slope beside a body of water)



NLTK incluye su propia implementación (`nltk.wsd.lesk`), que solo usa la definición sin ejemplos:

In [18]:
from nltk.wsd import lesk

for oracion in ["I went to the bank to deposit money and pay my loan",
                "We sat on the bank of the river and watched the water flow"]:
    sentido = lesk(word_tokenize(oracion), "bank", pos="n")
    print(f"{sentido.name():<20} {sentido.definition()}")

depository_financial_institution.n.01 a financial institution that accepts deposits and channels the money into lending activities
bank.n.01            sloping land (especially the slope beside a body of water)


**Limitaciones de Lesk:**

- Las definiciones de diccionario son cortas, así que los solapamientos son pequeños y frágiles (un empate se decide por el orden de los sentidos).
- No considera sinónimos: *money* y *cash* no cuentan como coincidencia.
- Depende de tener un recurso como WordNet, que es mucho más pobre para el español.

Los métodos modernos resuelven la ambigüedad de forma implícita: los **embeddings contextuales** de modelos como BERT (módulo 4) asignan vectores distintos a *banco* según la oración en que aparece. Aun así, Lesk ilustra la idea central que retomaremos en el cuaderno 05: **el significado de una palabra se infiere de las palabras que la rodean.**

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Bird, S., Klein, E. & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media. [nltk.org/book](https://www.nltk.org/book/)
- Documentación de spaCy. [spacy.io/usage](https://spacy.io/usage)
- Lesk, M. (1986). Automatic sense disambiguation using machine readable dictionaries. *SIGDOC '86*.
- Universal Dependencies: etiquetas POS universales. [universaldependencies.org/u/pos](https://universaldependencies.org/u/pos/)
- Miller, G. A. (1995). WordNet: a lexical database for English. *Communications of the ACM*, 38(11), 39–41.